In [138]:
import os
os.environ["OPENAI_API_KEY"] = "sk-1da2fbebf4b94353b76002c873c5ebab"

from openai import OpenAI
client = OpenAI(
    base_url = "https://api.deepseek.com"
)

In [139]:
def load_document(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

document = load_document("sample_prd.txt")

In [140]:
BASE_PROMPT = """
    【角色】
    你是一名资深 AI 产品分析助手，擅长从产品文档中提炼关键信息。

    【产品文档】
    ---
    {document}
    ---

    【用户问题】
    {question}
    """

TASK_PROMPTS = {
    "summary": {
        "task_instruction": """
        请按以下步骤完成任务（只输出最终结果）：
        - 步骤 1：识别产品的核心功能模块
        - 步骤 2：筛选最重要的 3 个功能
        - 步骤 3：用简洁语言总结
        现在请给出最终总结结果。
        """,
        "output_requirement": """
        请严格按照以下 JSON Schema 输出，不要添加任何多余文字。
        Schema:
        {
            "task": "summary",
            "points": [
                {
                "title": "string",
                "description": "string"
                }
            ]
        }
        要求：
        - points 中必须包含 3 个元素
        - title 简短（不超过 10 个字）
        - description 简要说明功能
        """
    },
    "risk": {
        "task_instruction": """
        在回答前，请先在内部制定一个简短的分析计划，然后基于该计划给出最终的改进建议。
        注意：
        - 不要输出分析计划
        - 只输出最终建议
        """,
        "output_requirement": """
        请严格按照以下 JSON Schema 输出，不要添加任何多余文字。
        Schema:
        {
            "task": "risk",
            "risks": 
            [
                {
                "risk": "string",
                "impact": "string"
                }
            ]
        }
        要求：
        - 从产品或使用角度分析
        - 输出 3 条主要风险
        - risk 简短（不超过 10 个字）
        """
    },
    "advice": {
        "task_instruction": """ArithmeticError
        请基于产品文档内容，提出改进建议，并以 JSON 格式输出。
        """,
        "output_requirement": """
        请严格按照以下 JSON Schema 输出，不要添加任何多余文字。
        Schema:
        {
            "task": "advice",
            "advices": [
                {
                    "advice": "string",
                    "description": "string"
                }
            ]
        }
        要求：
        - 建议要具体、可执行
        - 输出 3 条
        - advice 简短（不超过 10 个字）
        """
    }
}

SYSTEM_JSON = """
    你是一个严格的 AI 接口实现助手。
    你必须：
    - 只输出 JSON
    - 不使用 Markdown
    - 不添加任何解释性文字
    - 确保 JSON 可被程序直接解析
    """

SYSTEM_REASONING = """
    你是一名专业的 AI 产品分析助手。

    在回答前，你需要在内部完成以下步骤：
    1. 通读并理解产品文档
    2. 提取与当前任务最相关的信息
    3. 组织清晰、有逻辑的结果

    注意：
    - 不要输出你的分析过程
    - 只输出最终答案
    - 严格遵守输出格式要求
    """

In [141]:
def build_prompt(task: str, question: str, document: str) -> str:
    task_conf = TASK_PROMPTS[task]
    return f"""
    {BASE_PROMPT.format(document=document, question=question)}

    【当前任务】
    {task_conf["task_instruction"]}

    【输出要求】
    {task_conf["output_requirement"]}
    """

In [142]:
def ask_llm(prompt: str, system_prompt: str = "你是一个 AI 产品助手", temperature = 0.5):
    response = client.chat.completions.create(
        model = "deepseek-chat",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature = temperature,
        max_tokens = 800
    )
    return response.choices[0].message.content.strip()

In [143]:
QUESTION = "请对文档进行总结"
TASK = "summary"

prompt_A = build_prompt(TASK, QUESTION, document)
print("=== Prompt A ===")
print(ask_llm(prompt_A, system_prompt=SYSTEM_REASONING))

=== Prompt A ===
{
    "task": "summary",
    "points": [
        {
            "title": "多端协同考试管理",
            "description": "产品包含管理端（Web）、监考端和考试端（Windows客户端），形成完整的在线口语模考闭环。管理端负责全局组织，监考端负责考场执行，考试端供学生答题。"
        },
        {
            "title": "灵活的考试编排与执行",
            "description": "支持考试和模测两种模式，可精细编排考生场次考场，也可简化组织。监考端能自动下载考试数据、控制考试流程（允许登录、开始/结束考试）并实时监控所有考试端状态。"
        },
        {
            "title": "自动化部署与设备管控",
            "description": "通过监考端生成绑定本机IP的考试端安装包，实现考场内设备的快速、免配置部署。提供设备绑定/解绑、异常标记、远程寻找及状态监控等一站式设备管理功能。"
        }
    ]
}
